In [1]:
"""
Implementation of LeNet-5 [LeCun et al. 1998]
"Gradient-Based Learning Applied to Document Recognition"

Every design choice below is taken from the paper, NOT the modern equivalent:

    paper choice                       | modern replacement we're NOT using
    -----------------------------------+-----------------------------------
    scaled tanh activation             | ReLU / GELU
    trainable subsampling (S2, S4)     | max-pool / strided conv
    sparse channel connectivity in C3  | full connectivity
    Euclidean RBF output layer         | linear + softmax
    MAP / discriminative loss          | cross-entropy
    fully-connected C5 + F6 tail       | global average pooling
    fixed bipolar 7x12 prototypes      | learned dense classifier head
    no batch norm                      | BatchNorm / LayerNorm
    U[-2.4/fan_in, +2.4/fan_in] init   | He / Glorot init
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



# 1.  Activation function
#   f(a) = A * tanh(S * a),  with A = 1.7159 and S = 2/3.

A = 1.7159
S = 2.0 / 3.0


class ScaledTanh(nn.Module):
    def forward(self, x):
        return A * torch.tanh(S * x)



# 2.  Trainable subsampling layer S2 / S4
# Parameter counts:  S2 = 12 params (6 fmaps × 2), S4 = 32 params (16 fmaps × 2).

class Subsampling(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        self.num_features = num_features
        # Initialize to (coef=1, bias=0) so the initial map is just a sum over each 2×2 window; gradient descent will tune from there.
        self.coefficient = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.activation = ScaledTanh()

    def forward(self, x):
        # avg_pool2d gives the MEAN of the 2×2 window; multiplying by 4 turns it into the SUM, matching the paper's description literally.
        x = F.avg_pool2d(x, kernel_size=2, stride=2) * 4.0
        # Per-channel affine: one scalar coef and one scalar bias per fmap.
        x = x * self.coefficient.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)
        return self.activation(x)



# 3.  Sparse C3 connectivity
# The paper does NOT connect every C3 feature map to every S2 feature map.
# Instead each of the 16 C3 fmaps reads only a subset of the 6 S2 fmaps:

C3_TABLE = torch.tensor([
    # C3 fmap:  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15
    [          1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1 ],  # S2 fmap 0
    [          1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1 ],  # S2 fmap 1
    [          1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1 ],  # S2 fmap 2
    [          0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1 ],  # S2 fmap 3
    [          0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1 ],  # S2 fmap 4
    [          0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1 ],  # S2 fmap 5
], dtype=torch.float32)


class C3Sparse(nn.Module):
    """5×5 convolution with the paper's hand-designed sparse channel mask.

    Implementation note: we keep a dense (16,6,5,5) Conv2d kernel and zero
    out the disallowed (out_fmap, in_fmap) slots with a fixed binary mask
    on every forward pass. This is mathematically identical to having only
    60 separate kernels wired to specific input fmaps, and it lets us reuse
    a single fast cuDNN conv kernel instead of looping per output channel.

    Side effect: `sum(p.numel() for p in self.parameters())` will report
    16*6*25 + 16 = 2416, not the paper's 1516. The masked-out weights are
    physically allocated but receive zero gradient (the chain rule passes
    through the mask), so they stay frozen and contribute nothing to the
    output. Use `live_param_count` below for the paper's number.
    """

    def __init__(self):
        super().__init__()
        # Dense kernel (we will mask it). Shape (out=16, in=6, kH=5, kW=5).
        self.conv = nn.Conv2d(6, 16, kernel_size=5, bias=True)
        # mask shape (16, 6, 1, 1) so it broadcasts against (16, 6, 5, 5).
        mask = C3_TABLE.t().contiguous().view(16, 6, 1, 1)
        self.register_buffer("mask", mask)

    def forward(self, x):
        return F.conv2d(x, self.conv.weight * self.mask, self.conv.bias)

    @property
    def num_channel_connections(self) -> int:
        """Should report 60 — matches the paper."""
        return int(self.mask.sum().item())

    @property
    def live_param_count(self) -> int:
        """Effective trainable params — matches paper's 1516.

        Counts only weights that aren't masked out, plus all 16 biases.
        """
        weights_per_conn = self.conv.kernel_size[0] * self.conv.kernel_size[1]
        return self.num_channel_connections * weights_per_conn + self.conv.out_channels


# ===========================================================================
# 4.  RBF output layer with FIXED bipolar prototypes   (Section II-B)
# ===========================================================================
# Each of the 10 output units is a Euclidean Radial Basis Function:
#
#       y_i(x) = Σ_j (x_j − w_ij)²
#
# where x is the 84-dim F6 activation and w_i ∈ {−1, +1}^84 is the (FIXED,
# non-trainable) prototype of class i.
#
# Why 84? Because 7 × 12 = 84 — each prototype is a stylized 7-wide × 12-tall
# bipolar bitmap of the digit's glyph. The paper's intuition is that F6 is
# effectively "the network's attempt to draw the input as a 7×12 image", and
# a small RBF output means F6 has produced something close to one of the
# prototype glyphs.
#
# Why bipolar (±1) prototypes and not 0/1? Because F6 is itself squashed by
# the scaled tanh, so its outputs naturally live in roughly [−1.7, +1.7].
# Targeting ±1 matches that range; targeting 0/1 would put F6 in the wrong
# regime and could let activations saturate.

# Stylized 7×12 glyphs for digits 0..9.  '#' = +1, '.' = −1.
# These are NOT byte-identical to the paper's figures (those aren't published
# at pixel precision) but they follow the same recipe: hand-drawn digit-like
# shapes on a 7×12 grid.  You can edit these freely — the only constraint is
# that each glyph is 12 rows × 7 columns of '#' / '.'.
_DIGIT_BITMAPS = [
    # 0
    [".#####.",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     ".#####."],
    # 1
    ["...##..",
     "..###..",
     ".####..",
     "...##..",
     "...##..",
     "...##..",
     "...##..",
     "...##..",
     "...##..",
     "...##..",
     "...##..",
     ".######"],
    # 2
    [".#####.",
     "#.....#",
     "......#",
     "......#",
     ".....#.",
     "....#..",
     "...#...",
     "..#....",
     ".#.....",
     "#......",
     "#......",
     "#######"],
    # 3
    [".#####.",
     "#.....#",
     "......#",
     "......#",
     ".....#.",
     "..####.",
     ".....#.",
     "......#",
     "......#",
     "......#",
     "#.....#",
     ".#####."],
    # 4
    ["....##.",
     "...###.",
     "..#.##.",
     ".#..##.",
     "#...##.",
     "#...##.",
     "#######",
     "....##.",
     "....##.",
     "....##.",
     "....##.",
     "....##."],
    # 5
    ["#######",
     "#......",
     "#......",
     "#......",
     "######.",
     ".....#.",
     "......#",
     "......#",
     "......#",
     "......#",
     "#.....#",
     ".#####."],
    # 6
    ["..####.",
     ".#.....",
     "#......",
     "#......",
     "#......",
     "######.",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     ".#####."],
    # 7
    ["#######",
     "......#",
     "......#",
     ".....#.",
     ".....#.",
     "....#..",
     "....#..",
     "...#...",
     "...#...",
     "..#....",
     "..#....",
     ".#....."],
    # 8
    [".#####.",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     ".#####.",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     ".#####."],
    # 9
    [".#####.",
     "#.....#",
     "#.....#",
     "#.....#",
     "#.....#",
     ".######",
     "......#",
     "......#",
     "......#",
     "......#",
     "#.....#",
     ".#####."],
]


def _build_rbf_prototypes() -> torch.Tensor:
    """Turn the bitmap strings into a (10, 84) bipolar tensor."""
    prototypes = []
    for digit, glyph in enumerate(_DIGIT_BITMAPS):
        assert len(glyph) == 12, f"glyph {digit} must have 12 rows"
        pixels = []
        for row in glyph:
            assert len(row) == 7, f"glyph {digit} row '{row}' must have 7 cols"
            for ch in row:
                pixels.append(1.0 if ch == "#" else -1.0)
        prototypes.append(pixels)
    return torch.tensor(prototypes, dtype=torch.float32)  # (10, 84)


class EuclideanRBF(nn.Module):
    """Euclidean RBF output: squared distance from F6 to each of 10 prototypes.

    Crucially: the prototypes are stored as a *buffer*, not a Parameter, so
    they do NOT receive gradient updates. This matches the paper.
    """

    def __init__(self):
        super().__init__()
        self.register_buffer("prototypes", _build_rbf_prototypes())  # (10, 84)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x:           (N, 84)
        # prototypes: (10, 84)
        # We want:    y[n, i] = Σ_j (x[n, j] − w[i, j])² → (N, 10)
        diff = x.unsqueeze(1) - self.prototypes.unsqueeze(0)  # (N, 10, 84)
        return (diff ** 2).sum(dim=-1)                         # (N, 10)


# ===========================================================================
# 5.  The full LeNet-5 network   (Section II-B, Figure 2)
# ===========================================================================
#   Input :  1 × 32 × 32           (28×28 MNIST padded with 2px of background)
#   C1    :  6 fmaps,   5×5 conv         →  6 × 28 × 28
#   S2    :  6 fmaps,   trainable 2×2    →  6 × 14 × 14
#   C3    : 16 fmaps,   5×5 sparse conv  → 16 × 10 × 10
#   S4    : 16 fmaps,   trainable 2×2    → 16 ×  5 ×  5
#   C5    : 120 fmaps,  5×5 conv         → 120 ×  1 ×  1
#   F6    : 84 units,   fully connected  → 84
#   Out   : 10 Euclidean-RBF units       → 10  (squared distances)
#
# Trainable parameter total: 60 000   (paper's Table II).

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.act = ScaledTanh()

        self.c1 = nn.Conv2d(1, 6, kernel_size=5)      # 156  params
        self.s2 = Subsampling(6)                       # 12   params
        self.c3 = C3Sparse()                           # 1516 params
        self.s4 = Subsampling(16)                      # 32   params
        self.c5 = nn.Conv2d(16, 120, kernel_size=5)    # 48120 params
        self.f6 = nn.Linear(120, 84)                   # 10164 params
        self.out = EuclideanRBF()                      # 0     trainable

    def forward(self, x):                              # (N, 1, 32, 32)
        x = self.act(self.c1(x))                       # (N, 6, 28, 28)
        x = self.s2(x)                                  # activation is INSIDE
        x = self.act(self.c3(x))                        # (N, 16, 10, 10)
        x = self.s4(x)                                  # (N, 16, 5, 5)
        x = self.act(self.c5(x))                        # (N, 120, 1, 1)
        x = x.flatten(1)                                # (N, 120)
        x = self.act(self.f6(x))                        # (N, 84)
        return self.out(x)                              # (N, 10) — distances


# ===========================================================================
# 6.  MAP / discriminative loss   (Section II-C, eq. 9)
# ===========================================================================
# Per-example loss:
#
#       E = y_{Dp}  +  log( e^{-j}  +  Σ_i e^{-y_i} )
#
# where y_i are the RBF outputs and Dp is the correct-class index.
#
# Reading the two terms:
#   (a) y_{Dp}   — pulls the correct prototype's distance DOWN. By itself,
#                  this admits a trivial collapse: set all F6 outputs equal
#                  to a single (any) prototype, making every y_i tiny.
#   (b) the log-sum-exp term — pushes ALL distances up (softly). This breaks
#                  the collapse, because (a) only rewards the correct class.
#
# The constant j is the paper's "rubbish class" — a fixed competing logit
# that prevents the loss from drifting to −∞ when every y_i grows large.
# In effect, it's an 11th class with constant negative-distance logit −j.
#
# When you ignore j, this reduces exactly to cross-entropy on logits = −y_i.
# So the modern softmax + NLL story is the natural limit of the paper's loss.

def lenet5_loss(rbf_outputs: torch.Tensor,
                targets: torch.Tensor,
                j: float = 0.1) -> torch.Tensor:
    """
    rbf_outputs : (N, 10) squared distances (LeNet5's forward output).
    targets     : (N,)    integer class labels in 0..9.
    j           : rubbish-class constant from the paper.
    """
    N = rbf_outputs.size(0)
    # Term (a): pick out y_{Dp} for each example.
    y_correct = rbf_outputs.gather(1, targets.view(-1, 1)).squeeze(1)  # (N,)
    # Term (b): logsumexp over [−y_0, …, −y_9, −j].
    rubbish = torch.full((N, 1), -j,
                         device=rbf_outputs.device,
                         dtype=rbf_outputs.dtype)
    logits = torch.cat([-rbf_outputs, rubbish], dim=1)                # (N, 11)
    lse = torch.logsumexp(logits, dim=1)                              # (N,)
    return (y_correct + lse).mean()


# ===========================================================================
# 7.  Weight initialization   (Section III-C)
# ===========================================================================
# Paper quote: "weights are initialized with random values drawn from a
# uniform distribution between −2.4/F_i and +2.4/F_i, where F_i is the
# fan-in of the unit (the number of connections feeding into it)."
#
# This is the paper's version of what we today call "Glorot-like" init,
# predating Glorot 2010 by twelve years. Biases start at zero.

def lenet5_init(model: nn.Module) -> None:
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            fan_in = m.in_channels * m.kernel_size[0] * m.kernel_size[1]
            bound = 2.4 / fan_in
            nn.init.uniform_(m.weight, -bound, bound)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            fan_in = m.in_features
            bound = 2.4 / fan_in
            nn.init.uniform_(m.weight, -bound, bound)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        # Subsampling already sets coef=1, bias=0 in __init__, which is the
        # sensible starting point: identity-ish behavior (sum of 4 inputs).


# ===========================================================================
# 8.  Input preprocessing   (Section III)
# ===========================================================================
# Pad MNIST 28×28 → 32×32 so the 5×5 receptive field of C1 can be centered
# on every original pixel (including the corners). Then rescale pixel values
# so that background (white) = −0.1 and foreground (black) = +1.175 — which
# gives roughly zero mean and unit variance over the MNIST training set,
# as the paper notes.

INPUT_BG = -0.1
INPUT_FG = 1.175


def preprocess_mnist(images: torch.Tensor) -> torch.Tensor:
    """images : (N, 1, 28, 28) with pixels in [0, 1], 1 = digit foreground.
    Returns   : (N, 1, 32, 32) with bg = −0.1, fg = +1.175."""
    if images.dim() == 3:
        images = images.unsqueeze(1)
    images = F.pad(images, (2, 2, 2, 2), mode="constant", value=0.0)
    return images * (INPUT_FG - INPUT_BG) + INPUT_BG


# ===========================================================================
# 9.  Inference
# ===========================================================================
# Smaller RBF output = closer to that class's prototype = more confident.
# So the predicted class is the ARGMIN over the 10 outputs, not the argmax.

def predict(model: LeNet5, x: torch.Tensor) -> torch.Tensor:
    """x : (N, 1, 32, 32)  →  predicted class indices (N,)."""
    model.eval()
    with torch.no_grad():
        return model(x).argmin(dim=1)


# ===========================================================================
# 10. Smoke test
# ===========================================================================
if __name__ == "__main__":
    torch.manual_seed(0)

    model = LeNet5()
    lenet5_init(model)

    # 4 fake "images" so we can do a forward + backward pass.
    x = torch.randn(4, 1, 32, 32)
    y = torch.tensor([3, 7, 0, 1])

    rbf = model(x)
    loss = lenet5_loss(rbf, y)
    loss.backward()

    # Layer-by-layer parameter audit — compare to paper Table II.
    # Note: for C3 we report the *live* (unmasked) parameter count, since
    # the dense conv kernel includes 900 weights that are permanently masked
    # to zero and contribute nothing.
    def n(p):  # noqa: E306
        return sum(t.numel() for t in p)

    c3_live = model.c3.live_param_count
    c3_dead = n(model.c3.parameters()) - c3_live
    total_live = n(model.parameters()) - c3_dead

    print(f"Output shape         : {tuple(rbf.shape)}            (paper: (N, 10))")
    print(f"Loss                 : {loss.item():.4f}")
    print(f"C3 channel conns     : {model.c3.num_channel_connections}                  (paper: 60)")
    print(f"C1  trainable params : {n(model.c1.parameters()):>6}    (paper: 156)")
    print(f"S2  trainable params : {n(model.s2.parameters()):>6}    (paper: 12)")
    print(f"C3  live params      : {c3_live:>6}    (paper: 1516)")
    print(f"S4  trainable params : {n(model.s4.parameters()):>6}    (paper: 32)")
    print(f"C5  trainable params : {n(model.c5.parameters()):>6}    (paper: 48120)")
    print(f"F6  trainable params : {n(model.f6.parameters()):>6}    (paper: 10164)")
    print(f"Out trainable params : {n(model.out.parameters()):>6}    (paper: 0)")
    print(f"TOTAL live params    : {total_live:>6}    (paper: 60000)")
    print(f"Predicted classes    : {predict(model, x).tolist()}")

Output shape         : (4, 10)            (paper: (N, 10))
Loss                 : 83.9029
C3 channel conns     : 60                  (paper: 60)
C1  trainable params :    156    (paper: 156)
S2  trainable params :     12    (paper: 12)
C3  live params      :   1516    (paper: 1516)
S4  trainable params :     32    (paper: 32)
C5  trainable params :  48120    (paper: 48120)
F6  trainable params :  10164    (paper: 10164)
Out trainable params :      0    (paper: 0)
TOTAL live params    :  60000    (paper: 60000)
Predicted classes    : [4, 4, 0, 2]
